# Task 1: Data Cleaning & Preprocessing

## 1. Loading the Dataset

Loading the raw Netflix dataset using Pandas.

In [49]:
import pandas as pd
import numpy as np
from pathlib import Path

dataset_path = Path("../data/Dataset.csv")

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## 2. Dataset Structure Inspection

Inspecting column names, data types, non-null counts, and memory usage.


In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


## 3. Missing Values, Placeholders & Duplicates Inspection

In this step, we check data completeness and duplication across multiple levels before making any modifications:
- **Actual nulls (`NaN`)** and **blank/whitespace-only strings**.
- **Text placeholders** representing missing values (such as `"Not Given"` and `"Unknown"`).
- **Exact duplicate rows**, **`show_id` uniqueness**, and **repeated `(title, type)` records**.

In [51]:
obj_cols = df.select_dtypes(include=["object", "string"]).columns

profile = pd.DataFrame({
    "nulls": df[obj_cols].isna().sum(),
    "blanks": [int(df[c].astype(str).str.strip().eq("").sum()) for c in obj_cols],
    "unique": [int(df[c].nunique()) for c in obj_cols],
})
display(profile)

for col in obj_cols:
  if col == "show_id":
    continue
  vc = df[col].value_counts()
  print(f"\n=== {col} (unique={df[col].nunique()}) ===")
  print(vc.to_dict() if df[col].nunique() <= 20 else vc.head(5).to_dict())

,nulls,blanks,unique
show_id,0,0,8790
type,0,0,2
title,0,0,8787
director,0,0,4528
country,0,0,86
date_added,0,0,1713
rating,0,0,14
duration,0,0,220
listed_in,0,0,513



=== type (unique=2) ===
{'Movie': 6126, 'TV Show': 2664}

=== title (unique=8787) ===
{'9-Feb': 2, '15-Aug': 2, '22-Jul': 2, 'Dick Johnson Is Dead': 1, 'Ganglands': 1}

=== director (unique=4528) ===
{'Not Given': 2588, 'Rajiv Chilaka': 20, 'Raúl Campos, Jan Suter': 18, 'Alastair Fothergill': 18, 'Suhas Kadav': 16}

=== country (unique=86) ===
{'United States': 3240, 'India': 1057, 'United Kingdom': 638, 'Pakistan': 421, 'Not Given': 287}

=== date_added (unique=1713) ===
{'1/1/2020': 110, '11/1/2019': 91, '3/1/2018': 75, '12/31/2019': 74, '10/1/2018': 71}

=== rating (unique=14) ===
{'TV-MA': 3205, 'TV-14': 2157, 'TV-PG': 861, 'R': 799, 'PG-13': 490, 'TV-Y7': 333, 'TV-Y': 306, 'PG': 287, 'TV-G': 220, 'NR': 79, 'G': 41, 'TV-Y7-FV': 6, 'NC-17': 3, 'UR': 3}

=== duration (unique=220) ===
{'1 Season': 1791, '2 Seasons': 421, '3 Seasons': 198, '90 min': 152, '97 min': 146}

=== listed_in (unique=513) ===
{'Dramas, International Movies': 362, 'Documentaries': 359, 'Stand-Up Comedy': 334, '

### Scanning for Missing-Value Placeholders

In [52]:
missing_tokens = {
    "not given",
    "not available",
    "unknown",
    "missing",
    "none",
    "null",
    "n/a",
    "na",
    "tbd",
    "not specified",
    "not provided"
}

for col in df.select_dtypes(include=["object", "string"]).columns:
    s = df[col].dropna().astype(str).str.strip()
    hits = s[s.str.lower().isin(missing_tokens)]
    if not hits.empty:
        print(f"\n[{col}]")
        print(hits.value_counts())


[title]
title
Unknown    1
Name: count, dtype: int64

[director]
director
Not Given    2588
Name: count, dtype: int64

[country]
country
Not Given    287
Name: count, dtype: int64


In [53]:
norm_df = df.copy()
str_cols = norm_df.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
  norm_df[col] = norm_df[col].astype(str).str.strip().str.lower()

print("Exact duplicate rows (raw):", int(df.duplicated().sum()))
print("Exact duplicate rows (normalized):", int(norm_df.duplicated().sum()))
print("Unique show_id:", int(df["show_id"].nunique()), "out of", len(df))
print(
    "Duplicate (title, type) pairs (raw):",
    int(df.duplicated(subset=["title", "type"]).sum()),
)
print(
    "Duplicate (title, type) pairs (normalized):",
    int(norm_df.duplicated(subset=["title", "type"]).sum()),
)

dup_records = df[norm_df.duplicated(subset=["title", "type"], keep=False)].copy()
dup_records["_norm_title"] = norm_df.loc[dup_records.index, "title"]
dup_records = dup_records.sort_values(["_norm_title", "show_id"]).drop(
    columns="_norm_title"
)

display(
    dup_records[[
        "show_id",
        "type",
        "title",
        "director",
        "country",
        "date_added",
        "release_year",
        "duration",
    ]]
)

display(df[norm_df["title"] == "unknown"])

Exact duplicate rows (raw): 0
Exact duplicate rows (normalized): 0
Unique show_id: 8790 out of 8790
Duplicate (title, type) pairs (raw): 3
Duplicate (title, type) pairs (normalized): 7


,show_id,type,title,director,country,date_added,release_year,duration
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,124 min
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,124 min
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,144 min
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,144 min
393,s3997,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,1 Season
537,s5965,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,1 Season
2590,s3372,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,106 min
4718,s6530,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,106 min
415,s304,Movie,Esperando la carroza,Alejandro Doria,Argentina,8/5/2021,1985,95 min
4856,s6706,Movie,Esperando La Carroza,Alejandro Doria,Argentina,7/15/2018,1985,95 min


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
1336,s1468,Movie,Unknown,Jaume Collet-Serra,United Kingdom,1/1/2021,2011,PG-13,113 min,"Action & Adventure, Thrillers"


### Key Conclusions from Step 2

- **`"Unknown"` in `title` (`s1468`) is a valid movie title, not missing data:** Directed by Jaume Collet-Serra (2011) with complete metadata, leaving `"Not Given"` (in `director` and `country`) as the only true missing-value placeholder.
- **Date-like titles (`15-Aug`, `22-Jul`, `9-Feb`) are real titles, not column misalignment:** They reflect actual release titles (*15 August*, *22 July*, *9 February*) that were auto-formatted as short dates when the CSV was previously saved in spreadsheet software.
- **Text normalization uncovered 7 duplicate `(title, type)` pairs (14 rows) of two distinct kinds:** Four pairs share identical `date_added` values (including `Consequences`) while three pairs (`Esperando la carroza`, `Love in a Puff`, `Sin senos sí hay paraíso`) represent catalog re-additions with different `date_added` dates (2018/2019 vs. 2021).

## Step 3: Handling Missing Placeholders, Duplicates & Format Checks

Create `df_clean` from `df`, trim whitespace across string columns, convert `"Not Given"` in `director` and `country` to `pd.NA` (preserving the valid movie title `"Unknown"`), deduplicate normalized `(title, type)` pairs keeping the most recent `date_added`, and verify format consistency across `rating`, `date_added`, and `duration`.

In [54]:
df_clean = df.copy()

str_cols = df_clean.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
  df_clean[col] = df_clean[col].str.strip()

for col in ["director", "country"]:
  df_clean[col] = df_clean[col].replace("Not Given", pd.NA)

df_clean["_parsed_date"] = pd.to_datetime(
    df_clean["date_added"], errors="coerce"
)
df_clean["_norm_title"] = df_clean["title"].str.lower()
df_clean["_norm_type"] = df_clean["type"].str.lower()

df_clean = (
    df_clean.sort_values("_parsed_date", ascending=False)
    .drop_duplicates(subset=["_norm_title", "_norm_type"], keep="first")
    .sort_index()
    .drop(columns=["_parsed_date", "_norm_title", "_norm_type"])
    .reset_index(drop=True)
)

print("Shape after Step 3:", df_clean.shape)
print("Null counts:\n", df_clean.isna().sum().to_dict())
print(
    "Remaining normalized (title, type) duplicates:",
    int(
        (
            df_clean["title"].str.lower() + "||" + df_clean["type"].str.lower()
        ).duplicated().sum()
    ),
)
print(
    "Unparseable date_added:",
    int(pd.to_datetime(df_clean["date_added"], errors="coerce").isna().sum()),
)
print(
    "Movie duration format valid:",
    bool(
        df_clean.loc[df_clean["type"] == "Movie", "duration"]
        .str.match(r"^\d+\s+min$")
        .all()
    ),
)
print(
    "TV Show duration format valid:",
    bool(
        df_clean.loc[df_clean["type"] == "TV Show", "duration"]
        .str.match(r"^\d+\s+Seasons?$")
        .all()
    ),
)

Shape after Step 3: (8783, 10)
Null counts:
 {'show_id': 0, 'type': 0, 'title': 0, 'director': 2586, 'country': 287, 'date_added': 0, 'release_year': 0, 'rating': 0, 'duration': 0, 'listed_in': 0}
Remaining normalized (title, type) duplicates: 0
Unparseable date_added: 0
Movie duration format valid: True
TV Show duration format valid: True


## Step 4: Transforming Date, Categorical & Duration Columns

Convert `date_added` to `datetime` and extract `year_added` and `month_added`, cast `type` and `rating` to `category`, split `duration` into type-specific numeric columns (`duration_min` and `seasons`), and derive `primary_category` and `category_count` while keeping `listed_in` intact.

In [55]:
df_clean["date_added"] = pd.to_datetime(df_clean["date_added"])
df_clean["year_added"] = df_clean["date_added"].dt.year
df_clean["month_added"] = df_clean["date_added"].dt.month

df_clean["type"] = df_clean["type"].astype("category")
df_clean["rating"] = df_clean["rating"].astype("category")

duration_num = df_clean["duration"].str.extract(r"^(\d+)")[0].astype("Int64")
df_clean["duration_min"] = duration_num.where(df_clean["type"] == "Movie", pd.NA)
df_clean["seasons"] = duration_num.where(df_clean["type"] == "TV Show", pd.NA)

df_clean["primary_category"] = (df_clean["listed_in"].str.split(",").str[0].str.strip())
df_clean["category_count"] = df_clean["listed_in"].str.count(",") + 1

print("Shape after Step 4:", df_clean.shape)
print("\nData types:\n", df_clean.dtypes)
display(df_clean.head())

Shape after Step 4: (8783, 16)

Data types:
 show_id                        str
type                      category
title                          str
director                       str
country                        str
date_added          datetime64[us]
release_year                 int64
rating                    category
duration                       str
listed_in                      str
year_added                   int32
month_added                  int32
duration_min                 Int64
seasons                      Int64
primary_category            object
category_count               int64
dtype: object


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,duration_min,seasons,primary_category,category_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,90,<NA>,Documentaries,1
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,<NA>,1,Crime TV Shows,3
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,<NA>,1,TV Dramas,3
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,91,<NA>,Children & Family Movies,2
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,125,<NA>,Dramas,3


### Downstream Readiness Notes

- **Unit-safe duration split:** Separating `duration` into `duration_min` (6,121 movies) and `seasons` (2,662 TV shows) preserves the exact `8,783`-row partition while preventing invalid cross-format aggregations in EDA and forecasting.
- **Dual-purpose category representation:** While `country` already holds a single country per row, 6,771 of 8,783 titles span multiple categorys in `listed_in` keeping `listed_in` intact preserves full text context for Task 3 similarity modeling, while `primary_category` and `category_count` enable clean one-to-one grouping and complexity analysis in Task 2.

## Step 5: Exporting and Validating the Cleaned Dataset

Save `df_clean` to `../data/df_clean.csv` without the index column, then reload the exported file to verify its shape, column schema, null counts and duplicate-free integrity against the raw dataset.

In [56]:
clean_path = Path("../data/df_clean.csv")
df_clean.to_csv(clean_path, index=False)

df_verified = pd.read_csv(clean_path, parse_dates=["date_added"])

assert clean_path.exists(), "Exported file was not created."
assert (
    df_verified.shape == df_clean.shape
), f"Shape mismatch: {df_verified.shape} != {df_clean.shape}"
assert list(df_verified.columns) == list(
    df_clean.columns
), "Column names mismatch."
assert (
    df_verified.isna().sum().to_dict() == df_clean.isna().sum().to_dict()
), "Null counts mismatch."

norm_dups = int(
    (
        df_verified["title"].str.lower()
        + "||"
        + df_verified["type"].str.lower()
    ).duplicated().sum()
)
assert norm_dups == 0, f"Found {norm_dups} duplicate (title, type) records."

print(f"Saved successfully to: {clean_path.resolve()}")
print(f"File size: {clean_path.stat().st_size / 1024:.1f} KB")
print(f"Raw dataset shape (df): {df.shape}")
print(f"Cleaned dataset shape (df_verified): {df_verified.shape}")
print(f"Normalized (title, type) duplicates: {norm_dups}")
print("\nVerified null counts:\n", df_verified.isna().sum().to_dict())
display(df_verified.head())

Saved successfully to: /Users/ziad/Netflix-Content-Analytics/data/df_clean.csv
File size: 1256.4 KB
Raw dataset shape (df): (8790, 10)
Cleaned dataset shape (df_verified): (8783, 16)
Normalized (title, type) duplicates: 0

Verified null counts:
 {'show_id': 0, 'type': 0, 'title': 0, 'director': 2586, 'country': 287, 'date_added': 0, 'release_year': 0, 'rating': 0, 'duration': 0, 'listed_in': 0, 'year_added': 0, 'month_added': 0, 'duration_min': 2662, 'seasons': 6121, 'primary_category': 0, 'category_count': 0}


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,duration_min,seasons,primary_category,category_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,90.0,NaN,Documentaries,1
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,NaN,1.0,Crime TV Shows,3
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,NaN,1.0,TV Dramas,3
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,91.0,NaN,Children & Family Movies,2
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,125.0,NaN,Dramas,3


## Task 1 Summary

- **Key Cleaning Decisions Preserved:** `"Not Given"` in `director` and `country` was standardized to `NA`, while `"Unknown"` in `title` (`s1468`) was retained as a valid movie title; normalized `(title, type)` deduplication removed 7 duplicate pairs (keeping the latest `date_added`) without merging distinct works across formats (such as *Death Note* and *Fullmetal Alchemist*).
- **Downstream Schema Note (`Task 2`–`Task 4`):** Since CSV storage does not retain Pandas `category` or nullable `Int64` types (`duration_min` and `seasons` reload as `float64` due to `NaN`s), load `../data/df_clean.csv` with `parse_dates=["date_added"]` and cast `type` and `rating` to `category` when needed.